In [1]:
import sys; sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env")   # .env is in the project root, one level up

from google import genai
client = genai.Client()
resp = client.models.generate_content(model="gemini-3.5-flash", contents="Say hello in one word")
print(resp.text)

Hello


In [2]:
get_player_stats_declaration = {
    "name": "get_player_stats",
    "description": "Get a football player's season statistics (goals, assists, "
                   "minutes, etc.) from the EPL database. Use this whenever the "
                   "user asks about a specific player's stats.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string",
                     "description": "Player name, full or partial (e.g. 'Saka')"},
            "season": {"type": "string",
                       "description": "Season code like '2425'. Default is 2024-25."},
        },
        "required": ["name"],
    },
}

In [3]:
from google.genai import types
from data.loader import get_player_stats

MODEL = "gemini-3.5-flash"  

# ① Persona + output rules for the Judge
TEAM = "Tottenham"  # our club : change this one line to rebrand the whole scout

SYSTEM_PROMPT = f"""You are a scouting assistant for {TEAM}'s recruitment department.
Answer questions about player statistics using the get_player_stats tool.
Always base numbers on tool results, never on memory.
When a player played for multiple teams in one season, report each team's
numbers separately and clearly."""

# ② Register the tool from cell 2
tools = types.Tool(function_declarations=[get_player_stats_declaration])
config = types.GenerateContentConfig(
    tools=[tools],
    system_instruction=SYSTEM_PROMPT,
)

# ③ The question → decision → execution → answer loop
def ask_stats_agent(question):
    # Round 1: Gemini decides whether to call the tool
    resp = client.models.generate_content(
        model=MODEL, contents=question, config=config)

    part = resp.candidates[0].content.parts[0]
    if not part.function_call:
        return resp.text  # Gemini answered without the tool

    # Our code runs the function (Gemini never runs code itself)
    fc = part.function_call
    print(f"  [tool call] {fc.name}({dict(fc.args)})")   # observability
    result = get_player_stats(**fc.args)

    # Round 2: send the result back so Gemini can write the final answer
    follow_up = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Content(role="user", parts=[types.Part(text=question)]),
            resp.candidates[0].content,
            types.Content(role="tool", parts=[
                types.Part.from_function_response(
                    name="get_player_stats", response={"result": result})]),
        ],
        config=config)
    return follow_up.text

[07/06/26 22:41:24] INFO     No custom team name replacements found. You can configure these in       ]8;id=14949470;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=14949471;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jade/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=14949477;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=14949478;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/jade/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /Users/jade/soccerdata/data/FBref                 ]8;id=14949485;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=14949486;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

In [4]:
print(ask_stats_agent("How many goals did Saka score this season?"))
print("---")
print(ask_stats_agent("Son Heung-min left us last summer. How did he do in his final season?"))


[07/06/26 22:41:27] INFO     HTTP Request: POST                                                     ]8;id=14949493;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949494;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Saka'})


[07/06/26 22:41:28] INFO     HTTP Request: POST                                                     ]8;id=14949499;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949500;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

In the 2024-25 Premier League season, Bukayo Saka scored **6 goals** (including 1 penalty) for Arsenal. He also registered 10 assists in his 25 appearances (1,729 minutes).
---


[07/06/26 22:41:32] INFO     HTTP Request: POST                                                     ]8;id=14949505;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949506;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'season': '2324', 'name': 'Son Heung-min'})


[07/06/26 22:41:33] INFO     HTTP Request: POST                                                     ]8;id=14949511;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949512;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

                    WARNING  Warning: there are non-text parts in the response: ['function_call'],    ]8;id=14949519;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/types.py\types.py]8;;\:]8;id=14949520;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/google/genai/types.py#8121\8121]8;;\
                             returning concatenated text result from text parts. Check the full                    
                             candidates.content.parts accessor to get the full model response.                     

None


In [6]:
print(ask_stats_agent("We need a new left winger. How is Rashford performing?"))
print("---")
print(ask_stats_agent("Who is the best player in the world?"))


[07/06/26 22:43:15] INFO     HTTP Request: POST                                                     ]8;id=14949531;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949532;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Rashford'})


[07/06/26 22:43:21] INFO     HTTP Request: POST                                                     ]8;id=14949537;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949538;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Here is the scouting report on Marcus Rashford's performances for the 2024–25 season. 

Our database shows records for Rashford representing two different Premier League clubs this season. Here is the breakdown of his statistics for each side:

### **1. Performance for Manchester United**
* **Appearances:** 15 matches (12 starts)
* **Minutes Played:** 978 minutes
* **Goals:** 4 (all from open play)
* **Assists:** 1
* **Total Goal Contributions (G+A):** 5 (0.46 per 90 minutes)
* **Discipline:** 2 yellow cards, 0 red cards

### **2. Performance for Aston Villa**
* **Appearances:** 10 matches (4 starts)
* **Minutes Played:** 444 minutes
* **Goals:** 2 (1 penalty)
* **Assists:** 2
* **Total Goal Contributions (G+A):** 4 (0.81 per 90 minutes)
* **Discipline:** 0 yellow cards, 0 red cards

### **Scouting Summary**
Rashford has been highly efficient during his limited minutes at Aston Villa, averaging a goal involvement every 111 minutes. While his volume of starts and playing time has been g

[07/06/26 22:43:24] INFO     HTTP Request: POST                                                     ]8;id=14949543;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949544;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

As a scouting assistant for Tottenham's recruitment department, determining the "best player in the world" often depends on the specific profile, metrics, and tactical system we are looking at. 

If you have specific players in mind—whether it's our own **Son Heung-min**, or other Premier League standouts like **Erling Haaland**, **Mohamed Salah**, or **Cole Palmer**—I can pull up their official Premier League statistics (goals, assists, minutes played, and more) for comparison. 

Who would you like me to look up?


In [7]:
print(ask_stats_agent("Silva stats please"))

[07/06/26 22:43:25] INFO     HTTP Request: POST                                                     ]8;id=14949549;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949550;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Silva'})


                    INFO     HTTP Request: POST                                                     ]8;id=14949555;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14949556;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-fla                
                             sh:generateContent "HTTP/1.1 429 Too Many Requests"                                   

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 34.26245009s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'}, 